# Experiment E3 -- Class-Weighted Cross Entropy + Soft Dice (RQ-1)

**Thesis:** Segmentasi Zona Anatomi pada Citra MRI Pasien Kanker Prostat
menggunakan 2D U-Net dan Rekonstruksi 3D.

**RQ-1:** *Berapa Hasil Evaluasi dari Segmentasi Citra MRI pada Pasien Kanker
Prostat menggunakan 2D U-Net?* -- E3 is supporting evidence under RQ-1.

**Research question (E3):** *Does re-weighting the Cross Entropy term by median
class frequency -- so the minority PZ class contributes more -- improve validation
segmentation performance (in particular PZ recall) relative to E1, while keeping
every other training condition identical to E1?*

The outcome is **not presumed**. E3 may improve, match, or regress against E1;
the measured validation result decides, against thresholds fixed *before* the run.

---

## 1. Experiment overview

### The single experimental factor

```
E1:  Loss = 1.0 * CE(unweighted)          + 1.0 * SoftDice(CG, PZ)
E3:  Loss = 1.0 * CE(w=[0.0456,1,2.8477]) + 1.0 * SoftDice(CG, PZ)
                    ^^^^^^^^^^^^^^^^^^^^  <-- THE ONLY CHANGE
```

| Class | Weight |
|---|---|
| background (0) | 0.0456 |
| CG (1) | 1.0000 |
| PZ (2) | 2.8477 |

The weights go straight into `nn.CrossEntropyLoss(weight=w, reduction='mean')`,
i.e. PyTorch's **default weighted-mean normalization** `sum w_y l / sum w_y`.
**No manual 1/N re-normalization is applied** -- rescaling the CE term would
change the effective CE:Dice ratio and introduce a second independent variable.

### Why this factor

Voxel census over the 119 official training cases (this repository's own
measurement): background 94.20% | CG 4.30% | PZ 1.51%. Median class frequency
= 0.0430, so median-frequency weights are 0.0430 / f_c -> the table above.

E1 still shows **PZ precision 0.8073 >> PZ recall 0.7161** on the 20 validation
cases: PZ remains systematically *under-segmented* even with the foreground
soft-Dice term in place. E1's Dice term is region-based but balanced only across
the two foreground zones; its CE term is still purely pixel-wise and unweighted,
so 94.2% of the pixels it scores are background. E3 targets exactly that term.

### Controlled comparison

| | Baseline | E1 | E2 | **E3** |
|---|---|---|---|---|
| Loss | CE | 1.0 CE + 1.0 SoftDice | CE | **1.0 CE(weighted) + 1.0 SoftDice** |
| CE class weights | uniform | uniform | uniform | **[0.0456, 1.0, 2.8477]** |
| Dice term | none | SoftDice(CG,PZ), eps 1e-6 | none | **identical to E1** |
| Augmentation | none | none | ON | **none (= E1)** |
| Architecture | 2D U-Net (1 -> 3) | identical | identical | identical |
| Optimizer / LR / WD | AdamW / 1e-3 / 1e-4 | identical | identical | identical |
| Scheduler | none | none | none | none |
| Batch size / epochs / seed | 8 / 100 / 42 | identical | identical | identical |
| Precision | FP32 (AMP off) | identical | identical | identical |
| Preprocessing | per-volume norm, crop/pad 442 | identical | identical | identical |
| Slice sampling | weighted fg/bg sampler | identical | identical | identical |
| Validation | 20 official cases | identical | identical | identical |

**The single experimental factor changed from E1 to E3 is the CE CLASS WEIGHTS.**
`tests/test_e3_config.py` asserts that mechanically with an exhaustive key-by-key
diff against `configs/experiments/exp_e1_dice_ce.yaml`, and section 8 of this
notebook repeats that diff at runtime.

### Reference validation results (20 official validation cases)

| Arm | CG Dice | PZ Dice | Macro Dice | PZ recall | PZ precision |
|---|---|---|---|---|---|
| Baseline (CE) | 0.8642 | 0.7375 | 0.8008 | 0.7073 | 0.7855 |
| E1 (Dice+CE) | 0.8662 | 0.7527 | **0.8094** | 0.7161 | 0.8073 |
| E2 (CE + aug) | 0.8636 | 0.7327 | 0.7982 | 0.7008 | 0.7781 |
| **E3** | *this run* | *this run* | *this run* | *this run* | *this run* |

**E1 is the reference arm**, because E3 is nested in E1 (E3 = E1 + one factor).
E2 varies a *different* factor, so any E2-E3 difference is descriptive context.

### PRE-COMMITTED success criteria -- fixed before this run

| Category | Condition |
|---|---|
| **Improvement** | macro Dice >= 0.8194 **AND** CG Dice >= 0.8612 |
| **Mechanistic success** (metric-neutral) | PZ recall >= 0.7461 **AND** macro Dice within +/-0.010 of E1 |
| **Similar** | macro Dice within +/-0.010 of E1 **AND** PZ recall gain < 0.030 |
| **Regression** | macro Dice <= 0.7994 **OR** CG Dice drops > 0.005 vs E1 |

Applied in a fixed precedence: regression -> improvement -> mechanistic ->
similar -> indeterminate. The thresholds live in
`scripts/finalize_e3_report.py::E3_CRITERIA` and **must not be changed after
seeing the results**. A single metric moving in the right direction is not a
success.

**Selection vs comparison** (two different metrics, never conflated):

* *within-run checkpoint selection* -- the batch-level foreground-macro validation
  Dice proxy in `src/train.py::validate`, **unchanged** from Baseline / E1 / E2;
* *across-experiment comparison* -- per-case, volume-level, original-voxel-space
  **macro Dice = (CG + PZ) / 2** over the 20 official validation cases.

### Scope limits enforced by this notebook

* The official **19-case test set is NOT used** anywhere -- not for training, not
  for checkpoint selection, not for any comparison or decision.
* No external validation (PROSTATEx / MSD) is touched.
* **No final model is selected.**
* No post-processing, no TTA, no scheduler, no architecture change.
* The CG/PZ label mapping is left exactly as Baseline / E1 / E2 use it, and its
  UNVERIFIED status is recorded as a limitation in the generated report.

---

## How to run

1. `Runtime > Change runtime type > T4 GPU`
2. `Runtime > Run all`
3. Approve the Google Drive mount prompt when it appears.

That is the only manual interaction required.

### If Colab disconnects or the T4 session times out

Reconnect the runtime and **Run all** again. With the default `RESUME = "auto"`
the notebook detects `latest_checkpoint.pt` on Drive, restores the model,
optimizer, RNG and sampler state, reloads the complete epoch history, and
continues from the next epoch. **Training never restarts from epoch 1, and no
previously recorded epoch is ever lost.** If the run is already at 100 epochs the
notebook skips training entirely and goes straight to evaluation and reporting.


## 2. Reproducibility configuration

`RESUME` is the only switch you may need to touch, and normally you do not:

| Value | Behaviour |
|---|---|
| `"auto"` **(default)** | Resume if a usable checkpoint exists, start fresh if not, skip training if already at 100 epochs, and stop with a clear error if the checkpoint is corrupt. Correct for both the first run and every re-run. |
| `True` | Require a checkpoint. Fail loudly if there is none (protects against silently restarting a long run from scratch). |
| `False` | Require a clean directory. Fail loudly if a checkpoint or history already exists (protects an existing run from being overwritten). |

Everything else below is fixed by the committed config and must not be edited here.


In [ ]:
# ============================== THE ONLY SWITCH ==============================
RESUME = 'auto'      # 'auto' (recommended) | True (must resume) | False (must be fresh)
# =============================================================================

assert RESUME in ('auto', True, False), "RESUME must be 'auto', True, or False"

# Locations. These follow the existing project convention used by the
# Baseline / E1 / E2 notebooks: <DRIVE_ROOT>/results/<experiment name>/.
DRIVE_ROOT          = '/content/drive/MyDrive/THESIS_PROSTATE158'
REPO_URL            = 'https://github.com/amadeussandro/prostate-mri-segmentation-thesis.git'
REPO_DIR            = '/content/thesis-project'
REPO_BRANCH         = 'main'
EXPERIMENT_NAME     = 'exp_e3_dice_ce_pzweight'
E3_CONFIG_RELPATH   = 'configs/experiments/exp_e3_dice_ce_pzweight.yaml'
E1_CONFIG_RELPATH   = 'configs/experiments/exp_e1_dice_ce.yaml'

# THE independent variable. Asserted against the committed config, never applied
# to it -- so a silent drift in either place stops the run.
EXPECTED_CE_CLASS_WEIGHTS = [0.0456, 1.0000, 2.8477]   # index 0 = bg, 1 = CG, 2 = PZ

# Invariants asserted against the committed config later in this notebook.
EXPECTED = {
    'architecture'  : 'ProstateUNet2D',
    'in_channels'   : 1,
    'out_channels'  : 3,
    'init_features' : 32,
    'dropout_rate'  : 0.2,
    'loss'          : 'dice_ce',
    'ce_weight'     : 1.0,
    'dice_weight'   : 1.0,
    'dice_classes'  : [1, 2],
    'epsilon'       : 1e-6,
    'optimizer'     : 'AdamW',
    'learning_rate' : 0.001,
    'weight_decay'  : 0.0001,
    'batch_size'    : 8,
    'num_epochs'    : 100,
    'seed'          : 42,
    'amp'           : False,
    'target_mask'   : 't2_anatomy_reader1.nii.gz',
    'normalization' : 'per_volume',
    'z_resample'    : False,
    'spatial_mode'  : 'crop_pad',
    'target_size'   : [442, 442],
    'augmentation'  : False,
}

# Published anchors for the comparison table (validation, 20 cases).
BASELINE_REFERENCE = {'cg': 0.8642, 'pz': 0.7375, 'macro': 0.8008,
                      'pz_recall': 0.7073, 'pz_precision': 0.7855}
E1_REFERENCE       = {'cg': 0.8662, 'pz': 0.7527, 'macro': 0.8094,
                      'pz_recall': 0.7161, 'pz_precision': 0.8073}
E2_REFERENCE       = {'cg': 0.8636, 'pz': 0.7327, 'macro': 0.7982,
                      'pz_recall': 0.7008, 'pz_precision': 0.7781}

# PRE-COMMITTED success criteria. Mirrored from
# scripts/finalize_e3_report.py::E3_CRITERIA, which is the authoritative copy.
# DO NOT EDIT after seeing the results.
E3_CRITERIA = {
    'improvement'         : {'macro_dice_min': 0.8194, 'cg_dice_min': 0.8612},
    'mechanistic_success' : {'pz_recall_min': 0.7461, 'macro_dice_tolerance': 0.010},
    'similar'             : {'macro_dice_tolerance': 0.010, 'pz_recall_gain_below': 0.030},
    'regression'          : {'macro_dice_max': 0.7994, 'cg_dice_max_drop_vs_e1': 0.005},
}

print(f'RESUME mode       : {RESUME!r}')
print(f'Experiment        : {EXPERIMENT_NAME}')
print(f'Drive root        : {DRIVE_ROOT}')
print(f'Config (committed): {E3_CONFIG_RELPATH}')
print(f'CE class weights  : {EXPECTED_CE_CLASS_WEIGHTS}  (bg, CG, PZ)')


In [ ]:
# Global seeding happens inside src.train.run_training (src/utils.set_seed).
# Seeding here as well makes the notebook's own verification cells (sections 9-13)
# reproducible too; it cannot affect training, which re-seeds.
import os, random
SEED = EXPECTED['seed']
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
print(f'Notebook-level seed: {SEED}')


## 3. Colab environment / GPU verification and dependencies

Hard-fails rather than silently training a 100-epoch run on CPU.


In [ ]:
import subprocess
subprocess.run(['nvidia-smi'])


In [ ]:
import torch

CUDA_AVAILABLE = torch.cuda.is_available()
GPU_NAME = torch.cuda.get_device_name(0) if CUDA_AVAILABLE else None
DEVICE = 'cuda' if CUDA_AVAILABLE else 'cpu'

print(f'CUDA available: {CUDA_AVAILABLE}')
print(f'GPU: {GPU_NAME}')
print(f'Device being used: {DEVICE}')
print(f'torch: {torch.__version__}')

if not CUDA_AVAILABLE:
    raise RuntimeError(
        'No CUDA GPU detected. A full 100-epoch E3 run must NOT fall back to CPU.\n'
        'Fix: Runtime > Change runtime type > T4 GPU, then Runtime > Run all.'
    )
if GPU_NAME and 'T4' not in GPU_NAME:
    print(f'\nNOTE: expected a T4; got {GPU_NAME}. Training will still run, but'
          ' epoch timings will not be comparable to the T4 reference.')


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'nibabel', 'scipy', 'pandas', 'pyyaml', 'matplotlib'], check=True)
import nibabel, scipy, pandas, yaml, matplotlib
print('nibabel', nibabel.__version__, '| scipy', scipy.__version__,
      '| pandas', pandas.__version__, '| matplotlib', matplotlib.__version__)
print()
print('SciPy is present, so the paired Wilcoxon signed-rank test in section 22')
print('will run. (The report degrades gracefully without it, but do not skip it.)')


## 4. Mount Google Drive

**This is the core of the resume safety.** Every piece of permanent experiment
state -- checkpoints, per-epoch metrics, history, run state, evaluation, report --
is written to Drive, never to Colab's ephemeral `/content`. When the VM is
recycled, nothing is lost and the next `Run all` continues where it stopped.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 5. Repository setup

Clones the repo on a fresh VM, pulls on a re-run (so a mid-run fix is picked up),
and records the exact commit the results were produced from. No local Windows
paths are used anywhere -- everything resolves from `REPO_DIR` and `DRIVE_ROOT`.


In [ ]:
import os, subprocess, sys

if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(['git', 'checkout', REPO_BRANCH], check=True)
subprocess.run(['git', 'pull', 'origin', REPO_BRANCH], check=True)

GIT_COMMIT = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'],
                            capture_output=True, text=True).stdout.strip()
print('Repo      :', REPO_DIR)
print('Branch    :', REPO_BRANCH)
print('HEAD      :', GIT_COMMIT)
subprocess.run(['git', '-C', REPO_DIR, 'log', '-1', '--oneline'], check=True)
print('\nWorking tree (should be clean in a fresh clone):')
subprocess.run(['git', '-C', REPO_DIR, 'status', '--short'], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

E3_CONFIG_REPO = os.path.join(REPO_DIR, E3_CONFIG_RELPATH)
E1_CONFIG_REPO = os.path.join(REPO_DIR, E1_CONFIG_RELPATH)
assert os.path.isfile(E3_CONFIG_REPO), f'Committed E3 config not found: {E3_CONFIG_REPO}'
assert os.path.isfile(E1_CONFIG_REPO), f'Committed E1 config not found: {E1_CONFIG_REPO}'
print('\nE3 config :', E3_CONFIG_REPO)
print('E1 config :', E1_CONFIG_REPO, '(reference arm, read-only)')


## 6. Dataset path validation

The Prostate158 train+validation archive is read from Drive and cached on the
local Colab disk (much faster random access during training), then symlinked into
the repo so the committed **relative** dataset paths resolve unchanged. The Drive
dataset location is configurable through `DRIVE_ROOT` in section 2 -- the existing
project mechanism, identical to the Baseline / E1 / E2 notebooks.


In [ ]:
import os, shutil, subprocess

DRIVE_DATASET_DIR = os.path.join(DRIVE_ROOT, 'dataset')
DRIVE_RESULTS_DIR = os.path.join(DRIVE_ROOT, 'results')
LOCAL_DATASET_DIR = '/content/local_dataset'
USE_LOCAL_DATASET_CACHE = True   # set False to read NIfTI directly from Drive

assert os.path.isdir(DRIVE_DATASET_DIR), (
    f'Dataset not found on Drive: {DRIVE_DATASET_DIR}\n'
    'Point DRIVE_ROOT (section 2) at the folder that contains dataset/ and results/.')
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

if USE_LOCAL_DATASET_CACHE:
    os.makedirs(LOCAL_DATASET_DIR, exist_ok=True)
    print('Syncing dataset Drive -> local (safe to re-run)...')
    rc = subprocess.run(['rsync', '-a', DRIVE_DATASET_DIR.rstrip('/') + '/',
                         LOCAL_DATASET_DIR.rstrip('/') + '/']).returncode
    if rc != 0:
        shutil.copytree(DRIVE_DATASET_DIR, LOCAL_DATASET_DIR, dirs_exist_ok=True)
    DATASET_SRC = LOCAL_DATASET_DIR
else:
    DATASET_SRC = DRIVE_DATASET_DIR

link_path = os.path.join(REPO_DIR, 'dataset')
if os.path.islink(link_path):
    os.remove(link_path)
elif os.path.exists(link_path):
    raise RuntimeError(f'{link_path} exists and is not a symlink -- refusing to overwrite.')
os.symlink(DATASET_SRC, link_path)
print('Linked', link_path, '->', os.readlink(link_path))

DS_ROOT   = os.path.join(REPO_DIR, 'dataset', 'prostate158_train', 'train')
TRAIN_CSV = os.path.join(REPO_DIR, 'dataset', 'prostate158_train', 'train.csv')
VALID_CSV = os.path.join(REPO_DIR, 'dataset', 'prostate158_train', 'valid.csv')
for p in (DS_ROOT, TRAIN_CSV, VALID_CSV):
    assert os.path.exists(p), f'Required dataset path missing: {p}'
print('\nDataset root :', DS_ROOT)
print('Train CSV    :', TRAIN_CSV)
print('Valid CSV    :', VALID_CSV)


## 7. Dataset integrity and scientific-control tests

Cache completeness, the official 119/20 split, and label validity. The official
19-case test archive must be **absent**, and the notebook asserts that.


In [ ]:
from src.data_integrity import format_cache_report, verify_dataset_cache

report = verify_dataset_cache(DS_ROOT, TRAIN_CSV, VALID_CSV)
print(format_cache_report(report))
assert bool(report['ok']), 'Dataset cache INCOMPLETE -- fix before training (see report above).'
print('\nDataset integrity: OK')


In [ ]:
import numpy as np, nibabel as nib
from src.splits import load_official_split

split = load_official_split(train_csv=TRAIN_CSV, valid_csv=VALID_CSV)
print(f'Official split -> train {len(split.train_ids)} | val {len(split.val_ids)} |',
      f'test {len(split.test_ids) if split.test_ids else 0} (test intentionally absent)')
assert len(split.train_ids) == 119, f'expected 119 training cases, got {len(split.train_ids)}'
assert len(split.val_ids) == 20, f'expected 20 validation cases, got {len(split.val_ids)}'
assert not (set(split.train_ids) & set(split.val_ids)), 'train/val overlap detected'
assert split.test_ids is None, 'the official test split must NOT be available to E3'

print('\nValidation cases (the SAME 20 used by Baseline, E1 and E2):')
print(' ', ', '.join(split.val_ids))

seen = set()
for pid in (split.train_ids[:3] + split.val_ids[:3]):
    m = nib.load(os.path.join(DS_ROOT, pid, 't2_anatomy_reader1.nii.gz')).get_fdata()
    seen |= set(int(round(v)) for v in np.unique(m))
    assert os.path.exists(os.path.join(DS_ROOT, pid, 't2.nii.gz')), f'missing t2 for {pid}'
print('\nObserved mask labels (sampled cases):', sorted(seen))
assert seen.issubset({0, 1, 2}), f'Unexpected labels: {seen}'
print('Label check: OK (0=background, 1=CG, 2=PZ under the UNVERIFIED mapping)')


### 7b. Scientific-control tests

Mechanically asserts that E3 differs from E1 **only** by `loss.ce_class_weights`,
that the class weights really reach `nn.CrossEntropyLoss`, that PyTorch's default
weighted-mean normalization is used, that Baseline / E1 / E2 are still bit-for-bit
reproducible, that history survives a resume, and that E3 cannot resume from a
Baseline / E1 / E2 checkpoint. If any of this has drifted, the run stops here.


In [ ]:
import subprocess, sys
rc = subprocess.run([sys.executable, '-m', 'pytest', '-q',
                     'tests/test_e3_config.py',
                     'tests/test_losses.py',
                     'tests/test_e1_config.py',
                     'tests/test_e2_config.py',
                     'tests/test_history_preservation.py',
                     'tests/test_resume.py',
                     'tests/test_checkpoint_planning.py'],
                    cwd=REPO_DIR).returncode
assert rc == 0, 'Scientific-control tests FAILED -- do not train until resolved.'
print('\nScientific control: VERIFIED')


## 8. Configuration loading and the E3 contract

The committed config `configs/experiments/exp_e3_dice_ce_pzweight.yaml` is the
source of truth. The **only** field overridden for Colab is
`experiment.output_dir`, pointed at Drive so everything survives a disconnect.

The decisive cell below re-runs the E1-vs-E3 key-by-key diff at runtime: the only
permitted differences are `experiment.{name,description,output_dir}` and
`loss.ce_class_weights`. Anything else stops the run.


In [ ]:
import copy, yaml
from src.augment import AugmentationConfig
from src.losses import DICE_CE, LossConfig
from src.utils import load_config

committed = load_config(E3_CONFIG_REPO)
committed_e1 = load_config(E1_CONFIG_REPO)


def _flatten(mapping, prefix=''):
    flat = {}
    for key, value in mapping.items():
        dotted = f'{prefix}{key}'
        if isinstance(value, dict):
            flat.update(_flatten(value, dotted + '.'))
        else:
            flat[dotted] = value
    return flat


ALLOWED_DEVIATIONS = {'experiment.name', 'experiment.description',
                      'experiment.output_dir', 'loss.ce_class_weights'}
flat_e1, flat_e3 = _flatten(committed_e1), _flatten(committed)
differing = {k for k in set(flat_e1) | set(flat_e3)
             if flat_e1.get(k, '<absent>') != flat_e3.get(k, '<absent>')}

print('=' * 82)
print('E1 -> E3 CONFIG DIFF (the single-factor check)')
print('=' * 82)
for key in sorted(differing):
    print(f'  {key}')
    print(f'      E1: {flat_e1.get(key, "<absent>")!r}')
    print(f'      E3: {flat_e3.get(key, "<absent>")!r}')
print('=' * 82)

unexpected = differing - ALLOWED_DEVIATIONS
assert not unexpected, (
    f'E3 deviates from E1 in key(s) that are NOT the approved independent variable: '
    f'{sorted(unexpected)}. E3 must be E1 + ce_class_weights only.')
assert 'loss.ce_class_weights' in differing, (
    'E3 does not actually differ from E1 -- the independent variable is missing.')
print('\nSINGLE-FACTOR CONFIRMED: E3 = E1 + loss.ce_class_weights, nothing else.')


In [ ]:
# ----- E3 experiment contract, asserted field by field -----
E3_DRIVE_DIR       = os.path.join(DRIVE_RESULTS_DIR, EXPERIMENT_NAME)
BASELINE_DRIVE_DIR = os.path.join(DRIVE_RESULTS_DIR, 'exp01_baseline')
E1_DRIVE_DIR       = os.path.join(DRIVE_RESULTS_DIR, 'exp_e1_dice_ce')
E2_DRIVE_DIR       = os.path.join(DRIVE_RESULTS_DIR, 'exp_e2_ce_aug')
os.makedirs(E3_DRIVE_DIR, exist_ok=True)

cfg = copy.deepcopy(committed)
cfg['experiment']['output_dir'] = E3_DRIVE_DIR

# Guard: output_dir is the ONLY permitted deviation from the committed config.
changed = sorted(k for k in set(committed) | set(cfg) if committed.get(k) != cfg.get(k))
assert changed == ['experiment'], f'Unexpected config deviation: {changed}'
for k in set(committed['experiment']) | set(cfg['experiment']):
    if k != 'output_dir':
        assert committed['experiment'].get(k) == cfg['experiment'].get(k), k

loss_spec = LossConfig.from_config(cfg)

contract = []
def require(label, actual, expected):
    ok = actual == expected
    contract.append((label, ok, f'{actual!r}'))
    print(f"[{'OK  ' if ok else 'FAIL'}] {label:<36}: {actual!r}"
          f"{'' if ok else f'   <-- EXPECTED {expected!r}'}")

require('model.architecture',    cfg['model']['architecture'],    EXPECTED['architecture'])
require('model.in_channels',     cfg['model']['in_channels'],     EXPECTED['in_channels'])
require('model.out_channels',    cfg['model']['out_channels'],    EXPECTED['out_channels'])
require('model.init_features',   cfg['model']['init_features'],   EXPECTED['init_features'])
require('model.dropout_rate',    cfg['model']['dropout_rate'],    EXPECTED['dropout_rate'])
require('loss.name',             loss_spec.name,                  EXPECTED['loss'])
require('loss.ce_weight',        loss_spec.ce_weight,             EXPECTED['ce_weight'])
require('loss.dice_weight',      loss_spec.dice_weight,           EXPECTED['dice_weight'])
require('loss.dice_classes',     list(loss_spec.dice_classes),    EXPECTED['dice_classes'])
require('loss.epsilon',          loss_spec.epsilon,               EXPECTED['epsilon'])
require('loss.ce_class_weights', [round(w, 4) for w in loss_spec.ce_class_weights],
        [round(w, 4) for w in EXPECTED_CE_CLASS_WEIGHTS])
require('training.optimizer',    cfg['training']['optimizer'],    EXPECTED['optimizer'])
require('training.learning_rate', cfg['training']['learning_rate'], EXPECTED['learning_rate'])
require('training.weight_decay', cfg['training']['weight_decay'], EXPECTED['weight_decay'])
require('training.batch_size',   cfg['training']['batch_size'],   EXPECTED['batch_size'])
require('training.num_epochs',   cfg['training']['num_epochs'],   EXPECTED['num_epochs'])
require('training.seed',         cfg['training']['seed'],         EXPECTED['seed'])
require('training.amp (FP32)',   bool(cfg['training'].get('amp', False)), EXPECTED['amp'])
require('scheduler',             'lr_scheduler' in cfg['training'], False)
require('early_stopping',        'early_stopping' in cfg['training'], False)
require('data.target_mask',      cfg['data']['target_mask'],      EXPECTED['target_mask'])
require('preproc.normalization', cfg['preprocessing']['normalization'], EXPECTED['normalization'])
require('preproc.z_resample',    cfg['preprocessing']['z_resample'], EXPECTED['z_resample'])
require('preproc.spatial_mode',  cfg['preprocessing']['spatial_mode'], EXPECTED['spatial_mode'])
require('preproc.target_size',   list(cfg['preprocessing']['target_size']), EXPECTED['target_size'])
require('augmentation.enabled',  AugmentationConfig.from_config(cfg).enabled, EXPECTED['augmentation'])
require('test split referenced', bool(cfg['data'].get('test_csv') or cfg['data'].get('test_dir')), False)

violations = [label for label, ok, _ in contract if not ok]
assert not violations, f'E3 CONTRACT VIOLATED: {violations}'
print(f'\nE3 contract: {len(contract)}/{len(contract)} invariants satisfied.')

E3_CONFIG_COLAB = '/content/config_e3_dice_ce_pzweight_colab.yaml'
with open(E3_CONFIG_COLAB, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False, default_flow_style=False)

NUM_EPOCHS   = cfg['training']['num_epochs']
LATEST_CKPT  = os.path.join(E3_DRIVE_DIR, 'latest_checkpoint.pt')
BEST_CKPT    = os.path.join(E3_DRIVE_DIR, 'best_model.pt')
METRICS_CSV  = os.path.join(E3_DRIVE_DIR, 'metrics.csv')
HISTORY_JSON = os.path.join(E3_DRIVE_DIR, 'training_history.json')
STATE_JSON   = os.path.join(E3_DRIVE_DIR, 'state', 'training_state.json')

print('\nE3 config (Colab):', E3_CONFIG_COLAB)
print('Drive output dir :', E3_DRIVE_DIR)
print('Max epochs       :', NUM_EPOCHS)


## 9. Dataset construction

`run_training` builds the real datasets. This cell verifies the construction and
preprocessing wiring on a small subset (a few cases) rather than materializing all
119 training volumes twice. Preprocessing is **identical to E1** -- per-volume
normalization, no z-resample, crop/pad to 442x442 -- and **no augmentation** is
attached to either split.


In [ ]:
import numpy as np
from src.train import _build_dataset
from src.transforms import PreprocessingConfig

probe_cfg = {k: (dict(v) if isinstance(v, dict) else v) for k, v in cfg.items()}
probe_cfg['data'] = dict(cfg['data'])
probe_cfg['data']['dataset_root'] = DS_ROOT

train_probe = _build_dataset(probe_cfg, split.train_ids[:2], augment=True)
val_probe   = _build_dataset(probe_cfg, split.val_ids[:1],   augment=False)

checks = []
def check(name, ok, detail=''):
    checks.append((name, bool(ok), detail))
    print(f"[{'PASS' if ok else 'FAIL'}] {name}{(' -- ' + str(detail)) if detail else ''}")

sample = train_probe[0]
print(f"Probe sample: case {sample['patient_id']} slice {sample['slice_idx']}")
print('  image', sample['image'].shape, sample['image'].dtype)
print('  mask ', sample['mask'].shape, sample['mask'].dtype,
      'labels', sorted(set(np.unique(sample['mask']).tolist())))

check('D1. image is single-channel 442x442', tuple(sample['image'].shape) == (1, 442, 442),
      tuple(sample['image'].shape))
check('D2. mask is 442x442 integer labels', tuple(sample['mask'].shape) == (442, 442)
      and sample['mask'].dtype == np.int64)
check('D3. labels subset of {0,1,2}',
      set(int(v) for v in np.unique(sample['mask'])).issubset({0, 1, 2}))
check('D4. image values finite', bool(np.isfinite(sample['image']).all()))
check('D5. NO augmentation on the training set (E3 = E1, aug OFF)',
      train_probe.transform is None)
check('D6. NO augmentation on the validation set', val_probe.transform is None)
check('D7. training samples are deterministic',
      np.array_equal(train_probe[0]['image'], train_probe[0]['image']))
# PreprocessingConfig is a dataclass, so `==` compares every resolved field.
check('D8. preprocessing matches E1 exactly',
      PreprocessingConfig.from_dict(cfg['preprocessing'])
      == PreprocessingConfig.from_dict(committed_e1['preprocessing']),
      PreprocessingConfig.from_dict(cfg['preprocessing']))
check('D9. no test data referenced',
      'test_csv' not in cfg['data'] and 'test_dir' not in cfg['data'])

failed = [n for n, ok, _ in checks if not ok]
assert not failed, f'Dataset construction checks FAILED: {failed}'
print(f'\nDataset construction: {len(checks)}/{len(checks)} checks passed.')


## 10. DataLoader construction

Training uses the seeded `WeightedRandomSampler` (foreground/background balancing),
**unchanged** from Baseline / E1 / E2; validation uses the complete, unweighted,
unshuffled slice population. The real loaders are built once, inside
`run_training`; this cell verifies the wiring on the small subset.


In [ ]:
from torch.utils.data import DataLoader
from src.train import _build_train_sampler, _dataloader_perf_kwargs

training_cfg = cfg['training']
print(f'Subset train slices: {len(train_probe)} | subset val slices: {len(val_probe)}')

sampler = _build_train_sampler(train_probe, EXPECTED['seed'],
                               training_cfg.get('use_weighted_sampling', True))
perf = _dataloader_perf_kwargs(training_cfg, DEVICE)
print('DataLoader perf kwargs:', perf)
assert sampler is not None, 'the weighted foreground/background sampler must be active'

train_loader = DataLoader(train_probe, batch_size=training_cfg['batch_size'],
                          shuffle=(sampler is None), sampler=sampler, **perf)
val_loader   = DataLoader(val_probe, batch_size=training_cfg['batch_size'],
                          shuffle=False, **perf)

batch = next(iter(train_loader))
print('\nTrain batch  : image', tuple(batch['image'].shape), batch['image'].dtype,
      '| mask', tuple(batch['mask'].shape), batch['mask'].dtype)
assert tuple(batch['image'].shape) == (training_cfg['batch_size'], 1, 442, 442)
assert tuple(batch['mask'].shape) == (training_cfg['batch_size'], 442, 442)
assert set(batch['mask'].unique().tolist()).issubset({0, 1, 2})
assert torch.isfinite(batch['image']).all()

vbatch = next(iter(val_loader))
print('Val batch    : image', tuple(vbatch['image'].shape), '| mask', tuple(vbatch['mask'].shape))
assert set(vbatch['mask'].unique().tolist()).issubset({0, 1, 2})

print('\nDataLoader wiring: OK (train weighted-sampled, val plain, neither augmented).')


## 11. Model creation

Built by `src/model.py::build_model` from the committed config -- the notebook
defines no architecture of its own. Asserted identical to Baseline / E1 / E2.


In [ ]:
from src.model import build_model

model_cfg = cfg['model']
for key in ('architecture', 'in_channels', 'out_channels', 'init_features', 'dropout_rate'):
    assert model_cfg[key] == EXPECTED[key], \
        f'model.{key} = {model_cfg[key]!r}, expected {EXPECTED[key]!r} (must match Baseline/E1/E2)'
    assert model_cfg[key] == committed_e1['model'][key], f'model.{key} differs from E1'

model = build_model(cfg).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Architecture : {model_cfg['architecture']}")
print(f"Input/output : {model_cfg['in_channels']} -> {model_cfg['out_channels']} channels")
print(f"Init features: {model_cfg['init_features']} | dropout {model_cfg['dropout_rate']}")
print(f'Parameters   : {n_params:,}')

dummy = torch.zeros(1, 1, 442, 442, device=DEVICE)
model.eval()
with torch.no_grad():
    logits = model(dummy)
assert tuple(logits.shape) == (1, 3, 442, 442), f'unexpected output shape {tuple(logits.shape)}'
assert torch.isfinite(logits).all()
print(f'Forward pass : OK, logits {tuple(logits.shape)}')
print('\nModel is identical to the Baseline/E1/E2 architecture (E3 changes nothing here).')


## 12. E3 loss creation -- the independent variable, verified before training

E3's objective is E1's compound loss with a **class-weighted** CE term:

```
Loss = 1.0 * CE(logits, target; weight=[0.0456, 1.0, 2.8477])
     + 1.0 * SoftDice(logits, target; classes=[1, 2], eps=1e-6)
```

The cells below prove, on synthetic tensors and then on a real training slice:

1. the weights actually reach `nn.CrossEntropyLoss(weight=...)`;
2. PyTorch's **default weighted-mean** normalization `sum w_y l / sum w_y` is used,
   and **not** a manual 1/N rescaling (which would change the CE:Dice ratio);
3. the Dice term is **numerically identical** to E1's, so only CE changed;
4. gradients are finite on the correct device and dtype;
5. the unweighted (Baseline / E1 / E2) paths are still bit-for-bit reproducible.


In [ ]:
from torch import nn
from src.losses import (CROSS_ENTROPY, DICE_CE, LossConfig, build_ce_class_weight_tensor,
                        compute_loss_components, soft_dice_loss)

loss_config = LossConfig.from_config(cfg)
e1_loss_config = LossConfig.from_config(committed_e1)

print('E3 loss spec  :', loss_config.to_dict())
print('E3 description:', loss_config.describe())
print('E1 loss spec  :', e1_loss_config.to_dict())
print()
assert loss_config.name == DICE_CE, f'E3 must use the compound Dice+CE loss, got {loss_config.name!r}'
assert loss_config.has_ce_class_weights, 'E3 must declare loss.ce_class_weights'
assert not e1_loss_config.has_ce_class_weights, 'E1 must remain an UNWEIGHTED CE'
assert len(loss_config.ce_class_weights) == cfg['model']['out_channels'], \
    'one CE weight per model output channel'
for i, expected in enumerate(EXPECTED_CE_CLASS_WEIGHTS):
    assert abs(loss_config.ce_class_weights[i] - expected) < 1e-9, \
        f'ce_class_weights[{i}] = {loss_config.ce_class_weights[i]}, expected {expected}'
# Every Dice parameter must be byte-for-byte E1's.
assert loss_config.ce_weight   == e1_loss_config.ce_weight
assert loss_config.dice_weight == e1_loss_config.dice_weight
assert loss_config.dice_classes == e1_loss_config.dice_classes == (1, 2)
assert loss_config.epsilon     == e1_loss_config.epsilon
print('Loss spec: E3 = E1 + ce_class_weights; every Dice parameter is unchanged.')
print('CE weighting :', loss_config.describe_ce_class_weights())


In [ ]:
# ---- Synthetic, deliberately class-imbalanced probe (like the real data) ----
torch.manual_seed(SEED)
probe_logits = torch.randn(4, 3, 32, 32, device=DEVICE)
probe_target = torch.zeros(4, 32, 32, dtype=torch.long, device=DEVICE)
probe_target[:, 4:20, 4:20] = 1     # CG block
probe_target[:, 22:28, 22:28] = 2   # small PZ block
frac = [(probe_target == c).float().mean().item() for c in (0, 1, 2)]
print(f'probe class fractions: bg {frac[0]:.3f} | CG {frac[1]:.3f} | PZ {frac[2]:.3f}')

loss_checks = []
def lcheck(name, ok, detail=''):
    loss_checks.append((name, bool(ok), detail))
    print(f"[{'PASS' if ok else 'FAIL'}] {name}{(' -- ' + str(detail)) if detail else ''}")

weight = torch.tensor(EXPECTED_CE_CLASS_WEIGHTS, dtype=torch.float32, device=DEVICE)
expected_ce   = nn.CrossEntropyLoss(weight=weight, reduction='mean')(probe_logits, probe_target)
expected_dice = soft_dice_loss(probe_logits, probe_target, class_ids=(1, 2), epsilon=1e-6)
expected_total = 1.0 * expected_ce + 1.0 * expected_dice

total, components = compute_loss_components(probe_logits, probe_target, loss_config)
e1_total, e1_components = compute_loss_components(probe_logits, probe_target, e1_loss_config)

lcheck('L1. total == 1.0*CE(weighted) + 1.0*SoftDice',
       torch.equal(total, expected_total), f'{total.item():.8f}')
lcheck('L2. CE term == nn.CrossEntropyLoss(weight=w)',
       abs(components['ce'] - expected_ce.item()) < 1e-6,
       f"{components['ce']:.8f}")

# PyTorch's weighted mean is sum_i w_{y_i} l_i / sum_i w_{y_i}; the 1/N variant differs.
per_pixel = nn.CrossEntropyLoss(reduction='none')(probe_logits, probe_target)
weight_map = weight[probe_target]
weighted_mean = ((weight_map * per_pixel).sum() / weight_map.sum()).item()
one_over_n    = (weight_map * per_pixel).mean().item()
lcheck('L3. uses PyTorch weighted mean  sum(w*l)/sum(w)',
       abs(components['ce'] - weighted_mean) < 1e-5, f'{weighted_mean:.8f}')
lcheck('L4. does NOT use a manual 1/N normalization',
       abs(components['ce'] - one_over_n) > 1e-4,
       f"1/N would give {one_over_n:.8f}, we compute {components['ce']:.8f}")

lcheck('L5. weighting CHANGED the CE term vs E1',
       abs(components['ce'] - e1_components['ce']) > 1e-6,
       f"E1 {e1_components['ce']:.6f} -> E3 {components['ce']:.6f}")
lcheck('L6. Dice term IDENTICAL to E1 (no second factor)',
       abs(components['dice'] - e1_components['dice']) < 1e-12,
       f"{components['dice']:.8f}")

# The weight tensor itself: right length, right dtype, right device.
w_tensor = build_ce_class_weight_tensor(loss_config.ce_class_weights, 3, device=probe_logits.device)
lcheck('L7. weight tensor dtype/device/length',
       w_tensor.dtype == torch.float32 and w_tensor.device.type == probe_logits.device.type
       and w_tensor.numel() == 3, f'{w_tensor.tolist()} on {w_tensor.device}')
lcheck('L8. PZ up-weighted, background down-weighted',
       w_tensor[2].item() > w_tensor[1].item() > w_tensor[0].item())

# Gradients.
grad_logits = probe_logits.clone().requires_grad_(True)
g_total, _ = compute_loss_components(grad_logits, probe_target, loss_config)
g_total.backward()
lcheck('L9. gradients finite and non-zero',
       bool(torch.isfinite(grad_logits.grad).all()) and float(grad_logits.grad.abs().sum()) > 0)
lcheck('L10. loss dtype is float32 (FP32 run)', g_total.dtype == torch.float32)

# Regression guard: the earlier arms must be untouched.
base_total, _ = compute_loss_components(probe_logits, probe_target,
                                        LossConfig(name=CROSS_ENTROPY))
lcheck('L11. Baseline CE still bit-identical to nn.CrossEntropyLoss()',
       torch.equal(base_total, nn.CrossEntropyLoss()(probe_logits, probe_target)))
lcheck('L12. E1 loss still bit-identical to unweighted CE + SoftDice',
       torch.equal(e1_total, nn.CrossEntropyLoss()(probe_logits, probe_target) + expected_dice))

# A length mismatch must be refused rather than silently weighting wrong classes.
try:
    compute_loss_components(probe_logits, probe_target,
                            LossConfig(name=DICE_CE, ce_class_weights=[1.0, 2.0]))
    lcheck('L13. class-count mismatch is refused', False, 'NOT refused')
except ValueError as exc:
    lcheck('L13. class-count mismatch is refused', 'ce_class_weights' in str(exc))

loss_failed = [n for n, ok, _ in loss_checks if not ok]
print(f'\nLoss verification: {len(loss_checks) - len(loss_failed)}/{len(loss_checks)} passed')
assert not loss_failed, f'E3 LOSS VERIFICATION FAILED: {loss_failed}'
print('The E3 loss is correct, class-weighted as specified, and E1-identical elsewhere.')


In [ ]:
# ---- The same check on a REAL training slice, so it is not only synthetic ----
real_index = next((i for i in range(len(train_probe))
                   if set(np.unique(train_probe[i]['mask']).tolist()) >= {0, 1, 2}),
                  len(train_probe) // 2)
real = train_probe[real_index]
real_image = torch.from_numpy(real['image']).unsqueeze(0).float().to(DEVICE)
real_target = torch.from_numpy(real['mask']).unsqueeze(0).long().to(DEVICE)
print(f"Real slice: case {real['patient_id']} slice {real['slice_idx']}")
voxels = real_target.numel()
for c, name in ((0, 'background'), (1, 'CG'), (2, 'PZ')):
    print(f'  {name:<11}: {(real_target == c).sum().item() / voxels * 100:6.2f}% of voxels')

model.eval()
with torch.no_grad():
    real_logits = model(real_image)
real_e3, real_e3_components = compute_loss_components(real_logits, real_target, loss_config)
real_e1, real_e1_components = compute_loss_components(real_logits, real_target, e1_loss_config)
print()
print(f"E1 (unweighted CE): total {real_e1.item():.6f} "
      f"= ce {real_e1_components['ce']:.6f} + dice {real_e1_components['dice']:.6f}")
print(f"E3 (weighted CE)  : total {real_e3.item():.6f} "
      f"= ce {real_e3_components['ce']:.6f} + dice {real_e3_components['dice']:.6f}")
assert abs(real_e3_components['dice'] - real_e1_components['dice']) < 1e-12, \
    'the Dice term differs on a real slice -- E3 would not be single-factor'
assert abs(real_e3_components['ce'] - real_e1_components['ce']) > 1e-9, \
    'the class weights had no effect on a real slice'
print()
print('On a real, strongly background-dominated slice the weighted CE differs from')
print('E1 while the Dice term is numerically identical. E3 is a single-factor change.')


## 13. Optimizer

AdamW, lr 1e-3, weight decay 1e-4, **no scheduler** -- identical to Baseline / E1 /
E2. Constructed inside `run_training`; verified here against the committed config
and exercised with one real optimizer step on a real slice.


In [ ]:
for key in ('optimizer', 'learning_rate', 'weight_decay', 'batch_size', 'num_epochs', 'seed'):
    assert training_cfg.get(key) == EXPECTED[key], \
        f'training.{key} = {training_cfg.get(key)!r}, expected {EXPECTED[key]!r}'
    assert training_cfg.get(key) == committed_e1['training'].get(key), \
        f'training.{key} differs from E1'
assert bool(training_cfg.get('amp', False)) is False, 'AMP must be OFF (FP32) for E3'
assert 'lr_scheduler' not in training_cfg and 'scheduler' not in training_cfg

optimizer = torch.optim.AdamW(model.parameters(),
                              lr=training_cfg['learning_rate'],
                              weight_decay=training_cfg['weight_decay'])
print(f"Optimizer    : {training_cfg['optimizer']}")
print(f"Learning rate: {training_cfg['learning_rate']} (constant)")
print(f"Weight decay : {training_cfg['weight_decay']}")
print( 'Scheduler    : NONE (constant learning rate)')
print( 'Precision    : FP32 (AMP off)')

model.train()
before = model.final_conv.weight.detach().clone()
optimizer.zero_grad(set_to_none=True)
step_logits = model(real_image)
step_loss, step_components = compute_loss_components(step_logits, real_target, loss_config)
step_loss.backward()
assert all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None), \
    'non-finite gradient on a real slice under the class-weighted loss'
optimizer.step()
assert not torch.equal(before, model.final_conv.weight.detach()), 'optimizer step had no effect'
print(f'\nBackward + step under the E3 loss: OK '
      f"(total {step_loss.item():.6f} | ce {step_components['ce']:.6f} "
      f"| dice {step_components['dice']:.6f})")

del model, optimizer, step_logits, step_loss, before, dummy, logits
del train_probe, val_probe, train_loader, val_loader, batch, vbatch, sampler
del real_image, real_target, real_logits, grad_logits, probe_logits, probe_target
torch.cuda.empty_cache()


## 14. Training configuration fingerprint

Printed here and persisted to Drive alongside the results by `run_training`, so the
exact settings a result came from -- including the CE class weights -- are always
recoverable from the artifacts alone.


In [ ]:
from src.utils import build_config_fingerprint, format_config_fingerprint

fingerprint = build_config_fingerprint(cfg, device=DEVICE)
print(format_config_fingerprint(fingerprint))

assert fingerprint['amp'] is False, 'AMP must be OFF for E3 (FP32 controlled comparison).'
assert fingerprint['precision'] == 'FP32'
assert fingerprint['seed'] == 42
assert fingerprint['num_epochs'] == 100
assert fingerprint['augmentation_enabled'] is False, 'E3 must not enable augmentation'
assert fingerprint['dice_classes'] == [1, 2]
assert fingerprint['ce_class_weights'] is not None, 'the E3 factor is missing from the fingerprint'
for i, expected in enumerate(EXPECTED_CE_CLASS_WEIGHTS):
    assert abs(fingerprint['ce_class_weights'][i] - expected) < 1e-9
assert fingerprint['references_test_split'] is False

e1_fingerprint = build_config_fingerprint(committed_e1, device=DEVICE)
assert e1_fingerprint['ce_class_weights'] is None, 'E1 must stay unweighted'

print('\nAMP: OFF | Precision: FP32 | Augmentation: OFF | Scheduler: OFF')
print('CE class weights:', fingerprint['ce_class_weights'], '(E1:', e1_fingerprint['ce_class_weights'], ')')
print('Test split referenced: NO')


## 15. Checkpoint / resume handling

E2 was interrupted by a Colab timeout. E3 is built so that costs nothing: the run
resumes from Drive and never restarts from epoch 1.

### What is persisted to Drive, and when

| Artifact | Written | Contents |
|---|---|---|
| `latest_checkpoint.pt` | **after every epoch**, atomically | model, optimizer, epoch, best metric, best epoch, full config (incl. `loss.ce_class_weights`), python/numpy/torch RNG state, sampler generator state |
| `best_model.pt` | whenever the validation proxy improves | model, optimizer, epoch, best metric, best epoch, config |
| `metrics.csv` | after every epoch, whole-file atomic rewrite | one row per epoch |
| `training_history.json` | after every epoch, atomically | richer per-epoch record + loss spec |
| `state/training_state.json` | after every epoch, atomically | last completed epoch, best epoch/metric, epochs remaining, resume flag |
| `config.yaml`, `config_fingerprint.{json,txt}` | at run start | the resolved configuration |

Every write goes through a temp file + `os.replace`, so a session killed mid-write
can never leave a truncated checkpoint or a half-written history.

### History is never overwritten

`run_training` reloads the existing history from Drive **before** the epoch loop and
merges new epochs into it keyed by epoch number
(`src/train.py::load_existing_history` / `upsert_epoch_record`). Resuming at epoch 57
therefore ends with epochs **1..100** on disk, not 57..100. Re-running an epoch that
already exists updates that one row in place instead of appending a duplicate.

### Configuration compatibility

`verify_checkpoint_compatibility` refuses to resume when the checkpoint's embedded
config disagrees with the current one on experiment name, architecture, **loss spec
including `ce_class_weights`**, augmentation, preprocessing, dataset split,
optimizer, LR, weight decay, batch size, seed or AMP. That is what makes it
impossible to continue E3 from the Baseline, E1 or E2 checkpoint -- an E1 checkpoint
is refused on the class weights alone, even if its name were forced to match.

### Epoch budget

Maximum 100 epochs. A checkpoint at epoch 63 resumes at 64; a checkpoint at 100 is
left alone and training is skipped. The **best** epoch is whichever epoch won the
validation proxy -- epoch 100 is never assumed to be the best model.


In [ ]:
import json
from src.train import inspect_checkpoint, load_existing_history, plan_training_run

info = inspect_checkpoint(LATEST_CKPT)
print('latest_checkpoint.pt ->')
for k in ('exists', 'readable', 'epoch', 'best_epoch', 'best_metric', 'resumable',
          'has_model_state', 'has_optimizer_state', 'has_rng_state', 'size_bytes', 'error'):
    print(f'  {k:<22}: {info.get(k)}')

existing_history = load_existing_history(HISTORY_JSON, METRICS_CSV)
recorded_epochs = [int(r['epoch']) for r in existing_history]

plan = plan_training_run(LATEST_CKPT, NUM_EPOCHS)
print()
print(plan['message'])

if plan['action'] == 'corrupt':
    raise RuntimeError('Existing E3 checkpoint is unusable -- see message above. '
                       'It has NOT been modified or deleted; investigate before re-running.')

# --- Reconcile the plan with the user's RESUME setting.
if RESUME is True and plan['action'] == 'fresh':
    raise RuntimeError(
        f'RESUME=True but no usable checkpoint exists at {LATEST_CKPT}.\n'
        'Refusing to silently restart a 100-epoch run from scratch. '
        'Set RESUME="auto" (or False) if a fresh run really is what you want.'
    )
if RESUME is False and plan['action'] in ('resume', 'already_complete'):
    raise RuntimeError(
        f"RESUME=False but an existing run was found at {E3_DRIVE_DIR} "
        f"(checkpoint epoch {plan['checkpoint_epoch']}, "
        f'{len(existing_history)} epoch record(s) on disk).\n'
        'Refusing to overwrite it. Set RESUME="auto" to continue that run, or move '
        'the directory aside first if you genuinely want to start over.'
    )

RESUME_FROM = plan['resume_from']
SKIP_TRAINING = plan['action'] == 'already_complete'

print()
print('=' * 78)
print('RESUME STATUS')
print('=' * 78)
print(f"  Checkpoint found        : {'YES -- ' + LATEST_CKPT if info['exists'] else 'NO'}")
print(f"  Action                  : {plan['action'].upper()}")
print(f"  Resuming from epoch     : {plan['start_epoch'] if plan['action'] == 'resume' else '-'}")
_best_epoch = plan.get('best_epoch')
_best_metric = plan.get('best_metric')
print(f"  Best epoch so far       : {_best_epoch if _best_epoch is not None else 'n/a'}")
_best_metric_text = f'{_best_metric:.6f}' if isinstance(_best_metric, float) else 'n/a'
print(f'  Best validation metric  : {_best_metric_text}')
print(f'  History length on disk  : {len(existing_history)} epoch record(s)'
      + (f' (epochs {min(recorded_epochs)}..{max(recorded_epochs)})' if recorded_epochs else ''))
print(f'  Epoch budget            : {NUM_EPOCHS}')
print(f'  Output directory        : {E3_DRIVE_DIR}')
print('=' * 78)

if os.path.isfile(STATE_JSON):
    with open(STATE_JSON) as f:
        print('\nPrevious run state:', json.dumps(json.load(f), indent=2)[:1200])


In [ ]:
# Explicit fresh-initialization guard: E3 must never start from another arm.
from src.train import verify_checkpoint_compatibility

for other_name, other_dir in (('baseline', BASELINE_DRIVE_DIR),
                              ('E1', E1_DRIVE_DIR),
                              ('E2', E2_DRIVE_DIR)):
    for filename in ('best_model.pt', 'latest_checkpoint.pt'):
        foreign = os.path.join(other_dir, filename)
        assert os.path.abspath(foreign) != os.path.abspath(RESUME_FROM or ''), \
            f'E3 must NOT resume from the {other_name} checkpoint'

if RESUME_FROM:
    ckpt_cfg = torch.load(RESUME_FROM, map_location='cpu', weights_only=False).get('config')
    result = verify_checkpoint_compatibility(ckpt_cfg, cfg, RESUME_FROM)
    if result['verified']:
        print('Checkpoint compatibility: VERIFIED -- same experiment, model, loss spec')
        print('  (including ce_class_weights), augmentation, preprocessing, split and')
        print('  hyperparameters.')
    else:
        print('Checkpoint compatibility: SKIPPED --', result['skipped_reason'])
    assert (ckpt_cfg or {}).get('experiment', {}).get('name') in (None, EXPERIMENT_NAME), \
        'the resume checkpoint belongs to a different experiment'
    ckpt_weights = ((ckpt_cfg or {}).get('loss') or {}).get('ce_class_weights')
    if ckpt_weights is not None:
        for i, expected in enumerate(EXPECTED_CE_CLASS_WEIGHTS):
            assert abs(float(ckpt_weights[i]) - expected) < 1e-9, \
                f'the resume checkpoint was trained with different CE weights: {ckpt_weights}'
        print('  Checkpoint CE class weights:', ckpt_weights, '-- match the E3 specification.')
else:
    print('FRESH INITIALIZATION: E3 starts from randomly initialized weights.')
    print('  It is NOT warm-started from the Baseline, E1 or E2 checkpoint.')


## 16. Training (max 100 epochs, FP32, class-weighted CE + SoftDice, auto-resume)

All state goes straight to Drive after every epoch. If this cell is interrupted,
reconnect the runtime and **Run all** again -- it continues at the next epoch and
the epoch history already on Drive is reloaded and appended to, never replaced.


In [ ]:
import time
from src.train import run_training

print('Experiment      :', EXPERIMENT_NAME)
print('Loss            :', loss_config.describe())
print('CE class weights:', list(loss_config.ce_class_weights), '(bg, CG, PZ)')
print('Augmentation    : OFF (same as E1)')
print('AMP             : OFF')
print('Precision       : FP32')
print(f'Device          : {DEVICE} ({GPU_NAME})')
print(f'Resume from     : {RESUME_FROM or "(fresh run)"}')
print()

if SKIP_TRAINING:
    print(f'E3 already completed {NUM_EPOCHS} epochs -- NOT retraining.')
    print('Continuing to evaluation and report generation.')
    train_summary = None
else:
    t0 = time.time()
    train_summary = run_training(
        E3_CONFIG_COLAB,
        resume_from=RESUME_FROM,
        history_path=HISTORY_JSON,
    )
    print(f'\nElapsed: {(time.time() - t0) / 3600.0:.2f} h')
    print()
    for k, v in train_summary.items():
        print(f'  {k:<20}: {v}')


In [ ]:
# History integrity check, immediately after training.
final_history = load_existing_history(HISTORY_JSON, METRICS_CSV)
final_epochs = [int(r['epoch']) for r in final_history]
print(f'Epoch records on disk: {len(final_history)}')
if final_epochs:
    print(f'Epoch range          : {min(final_epochs)}..{max(final_epochs)}')
    gaps = sorted(set(range(1, max(final_epochs) + 1)) - set(final_epochs))
    duplicates = len(final_epochs) != len(set(final_epochs))
    assert not duplicates, 'duplicate epoch rows found in the history'
    if gaps:
        print(f'WARNING: missing epoch record(s): {gaps}')
        print('  Training itself was continuous; these rows were never written and')
        print('  cannot be reconstructed.')
    else:
        print('History is COMPLETE and contiguous -- no epoch was lost across interruptions.')
if recorded_epochs:
    preserved = set(recorded_epochs).issubset(set(final_epochs))
    print(f'Pre-resume epochs preserved: {preserved} '
          f'({len(recorded_epochs)} before -> {len(final_epochs)} now)')
    assert preserved, 'HISTORY LOSS DETECTED: epochs recorded before this session are gone.'


## 17. Validation (the per-epoch signal)

The validation signal that ran inside training: the batch-level foreground-macro
Dice proxy, **unchanged** from Baseline / E1 / E2. This is the *selection* metric,
not the comparison metric. The validation set was never augmented, and the
validation loss uses the same class-weighted objective (reporting only -- it does
not change the selection rule).


In [ ]:
import pandas as pd

history_df = pd.read_csv(METRICS_CSV)
print(f'Per-epoch records: {len(history_df)}')
print('\nFirst 5 epochs:')
print(history_df.head(5).to_string(index=False))
print('\nLast 5 epochs:')
print(history_df.tail(5).to_string(index=False))

print('\nValidation proxy summary:')
print(f"  best   : {history_df['val_dice_proxy'].max():.6f} "
      f"(epoch {int(history_df.loc[history_df['val_dice_proxy'].idxmax(), 'epoch'])})")
print(f"  final  : {history_df['val_dice_proxy'].iloc[-1]:.6f} "
      f"(epoch {int(history_df['epoch'].iloc[-1])})")
print(f"  mean CG (slice-level): {history_df['val_dice_class1_CG'].tail(10).mean():.6f} (last 10 epochs)")
print(f"  mean PZ (slice-level): {history_df['val_dice_class2_PZ'].tail(10).mean():.6f} (last 10 epochs)")
print('\nLoss terms (last epoch):')
last = history_df.iloc[-1]
print(f"  train total {last['train_loss_total']:.6f} | train CE (weighted) "
      f"{last['train_loss_ce']:.6f} | train SoftDice {last['train_loss_dice']:.6f}")


## 18. Best-checkpoint selection

The evaluated model is the epoch that won the within-run validation proxy --
**not** automatically epoch 100. If the best epoch is 81, epoch 81 is the model.
The selection rule is byte-for-byte the one Baseline / E1 / E2 used.


In [ ]:
best_info = inspect_checkpoint(BEST_CKPT)
latest_info = inspect_checkpoint(LATEST_CKPT)

assert best_info['exists'] and best_info['readable'], f'No usable best_model.pt at {BEST_CKPT}'

proxy_best_epoch = int(history_df.loc[history_df['val_dice_proxy'].idxmax(), 'epoch'])
proxy_best_value = float(history_df['val_dice_proxy'].max())

print('=' * 78)
print('BEST CHECKPOINT SELECTION')
print('=' * 78)
print( '  Selection rule          : max batch-level foreground-macro val Dice proxy')
print( '                            (identical rule to Baseline, E1 and E2)')
print(f"  Best epoch (checkpoint) : {best_info['epoch']}")
print(f'  Best epoch (history)    : {proxy_best_epoch}')
print(f'  Best proxy value        : {proxy_best_value:.6f}')
print(f"  Last epoch trained      : {latest_info['epoch']}")
print(f"  Best == last epoch?     : {best_info['epoch'] == latest_info['epoch']}")
print(f'  best_model.pt           : {BEST_CKPT}')
print(f'  latest_checkpoint.pt    : {LATEST_CKPT}')
print('=' * 78)

if best_info['epoch'] != proxy_best_epoch:
    print('\nNOTE: the checkpoint epoch and the history argmax differ. This is expected',
          'only if an epoch record was lost; otherwise investigate before reporting.')
if best_info['epoch'] != latest_info['epoch']:
    print(f"\nThe best model is epoch {best_info['epoch']}, NOT the final epoch "
          f"{latest_info['epoch']}. That is a valid and expected outcome.")


## 19. Final validation evaluation (20 official validation cases)

Per-case, volume-level, original-voxel-space metrics -- Dice, IoU, HD95, ASD,
precision, recall -- for the selected checkpoint. Inference only, FP32, no
augmentation, no post-processing, no test-time augmentation.

**The official 19-case test set is not touched.** `split='val'` is hard-coded here,
and `scripts/finalize_e3_report.py` refuses outright to read a test-split summary.


In [ ]:
from src.evaluate import run_evaluation

eval_result = run_evaluation(
    config_path=E3_CONFIG_COLAB,
    checkpoint_path=BEST_CKPT,
    output_dir=E3_DRIVE_DIR,
    split='val',            # NEVER 'test' in E3
    class_ids=(0, 1, 2),
)
assert eval_result['metadata']['n_cases'] == 20, \
    f"expected 20 validation cases, evaluated {eval_result['metadata']['n_cases']}"
assert eval_result['metadata']['is_official_held_out_test'] is False
print('\nPer-case metrics written to:',
      os.path.join(E3_DRIVE_DIR, 'eval_val_per_case_metrics.csv'))


## 20. Per-case metrics

All 20 validation cases, every class, every metric -- the raw table the aggregate
numbers and the paired test are computed from.


In [ ]:
per_case = pd.read_csv(os.path.join(E3_DRIVE_DIR, 'eval_val_per_case_metrics.csv'))
print(f'Per-case rows: {len(per_case)}')
print('\nColumns:', list(per_case.columns))
print()
print(per_case.to_string(index=False))

# Per-case macro Dice, which is what the paired comparison in section 22 uses.
fg = per_case[per_case['class_id'].isin([1, 2])]
macro_per_case = fg.pivot(index='patient_id', columns='class_id', values='dice')
macro_per_case['macro_dice'] = macro_per_case[[1, 2]].mean(axis=1)
print('\nPer-case macro Dice:')
print(macro_per_case.to_string())
print(f"\nmacro Dice across cases: mean {macro_per_case['macro_dice'].mean():.4f} "
      f"+/- {macro_per_case['macro_dice'].std(ddof=1):.4f} SD (n={len(macro_per_case)})")


## 21. Aggregate metrics

Primary: CG Dice, PZ Dice, macro Dice. Secondary: IoU, HD95, ASD, precision,
recall -- all with mean, SD and the evaluation's bootstrap 95% CI.


In [ ]:
summary = eval_result['summary']
CG, PZ = 1, 2

e3_cg    = summary[CG]['dice']['mean']
e3_pz    = summary[PZ]['dice']['mean']
e3_macro = (e3_cg + e3_pz) / 2
e3_pz_recall    = summary[PZ]['recall']['mean']
e3_pz_precision = summary[PZ]['precision']['mean']
e3_macro_sd = float(macro_per_case['macro_dice'].std(ddof=1))

print('=' * 78)
n_eval_cases = eval_result['metadata']['n_cases']
print(f'E3 AGGREGATE VALIDATION METRICS ({n_eval_cases} cases, per-case, original voxel space)')
print('=' * 78)
print( '  PRIMARY')
print(f'    CG Dice    : {e3_cg:.4f}')
print(f'    PZ Dice    : {e3_pz:.4f}')
print(f'    MACRO Dice : {e3_macro:.4f}  (+/- {e3_macro_sd:.4f} SD across cases)')
print('-' * 78)
print( '  SECONDARY')
for class_id, label in ((CG, 'CG'), (PZ, 'PZ')):
    for metric in ('dice', 'iou', 'hd95_mm', 'asd_mm', 'precision', 'recall'):
        cell = summary[class_id].get(metric)
        if cell is None:
            continue
        sd = cell.get('std')
        sd_text = f' (SD {sd:.4f})' if isinstance(sd, float) else ''
        ci_low, ci_high = cell.get('ci95_low'), cell.get('ci95_high')
        ci_text = (f'  95%CI [{ci_low:.4f}, {ci_high:.4f}]'
                   if isinstance(ci_low, float) and isinstance(ci_high, float) else '')
        print(f'    {label} {metric:<10}: {cell["mean"]:.4f}{sd_text}{ci_text}')
print('=' * 78)


## 22. Comparison against Baseline / E1 / E2, and the pre-committed verdict

`scripts/finalize_e3_report.py` builds the four-arm table, the paired E3-vs-E1
statistics (bootstrap CI via the project's existing `src.metrics.bootstrap_ci`,
Wilcoxon signed-rank via SciPy) and applies the **pre-committed** success criteria.
No new statistical methodology is introduced for E3.

The Baseline / E1 / E2 `eval_val_summary.json` and per-case CSVs are read from
Drive -- they are generated artefacts and are intentionally not committed to Git.
**E1 is required**: without it there is no reference arm and no paired test.


In [ ]:
import subprocess, sys

e1_summary_path   = os.path.join(E1_DRIVE_DIR, 'eval_val_summary.json')
e1_per_case_path  = os.path.join(E1_DRIVE_DIR, 'eval_val_per_case_metrics.csv')
baseline_summary_path = os.path.join(BASELINE_DRIVE_DIR, 'eval_val_summary.json')
e2_summary_path   = os.path.join(E2_DRIVE_DIR, 'eval_val_summary.json')

if not os.path.isfile(e1_summary_path):
    raise FileNotFoundError(
        f'E1 validation summary not found on Drive: {e1_summary_path}\n'
        'E1 is the reference arm for E3 -- the comparison cannot be built without it. '
        'It is a generated artefact (gitignored): re-run the E1 validation evaluation, '
        'or copy the existing file to that path, then re-run this cell.')
if not os.path.isfile(e1_per_case_path):
    print(f'NOTE: E1 per-case CSV not found at {e1_per_case_path};')
    print('      the aggregate comparison will still be built, but the PAIRED')
    print('      Wilcoxon / won-lost analysis will be reported as unavailable.')
for label, path in (('Baseline', baseline_summary_path), ('E2', e2_summary_path)):
    if not os.path.isfile(path):
        print(f'NOTE: {label} summary not found at {path}; it will show as n/a.')

report_cmd = [sys.executable, 'scripts/finalize_e3_report.py',
              '--e3-dir', E3_DRIVE_DIR,
              '--e1-dir', E1_DRIVE_DIR,
              '--baseline-dir', BASELINE_DRIVE_DIR,
              '--e2-dir', E2_DRIVE_DIR]
rc = subprocess.run(report_cmd, cwd=REPO_DIR).returncode
assert rc == 0, 'E3 report generation failed.'


In [ ]:
# The four-arm comparison table, printed from the generated summary.json.
import json

with open(os.path.join(E3_DRIVE_DIR, 'summary.json'), encoding='utf-8') as f:
    e3_summary = json.load(f)

ARMS = ('baseline', 'e1', 'e2', 'e3')
ARM_LABELS = {'baseline': 'Baseline (CE)', 'e1': 'E1 (Dice+CE)',
              'e2': 'E2 (CE + aug)', 'e3': 'E3 (Dice+CE, weighted CE)'}
ARM_FACTOR = {'baseline': 'none (reference)', 'e1': '+ SoftDice(CG,PZ)',
              'e2': '+ augmentation', 'e3': '+ CE class weights'}

def cell(class_id, metric, arm):
    return e3_summary['per_class'][str(class_id)][metric][arm]

def f4(v):
    return '   n/a' if v is None else f'{v:6.4f}'

def f3(v):
    return '  n/a' if v is None else f'{v:5.3f}'

best_epochs = e3_summary.get('best_epochs_all_arms', {})
macro = e3_summary['macro_dice']

print('=' * 128)
print('BASELINE vs E1 vs E2 vs E3 -- validation, 20 cases, per-case volume-level, original voxel space')
print('=' * 128)
header = (f"{'Experiment':<26}{'Intervention':<22}{'BestEp':>7}{'CG Dice':>9}{'PZ Dice':>9}"
          f"{'Macro':>9}{'CGprec':>8}{'CGrec':>8}{'PZprec':>8}{'PZrec':>8}"
          f"{'CGHD95':>8}{'PZHD95':>8}{'CGASD':>7}{'PZASD':>7}")
print(header)
print('-' * 128)
for arm in ARMS:
    print(f"{ARM_LABELS[arm]:<26}{ARM_FACTOR[arm]:<22}"
          f"{str(best_epochs.get(arm, 'n/a')):>7}"
          f"{f4(cell(1, 'dice', arm)):>9}{f4(cell(2, 'dice', arm)):>9}"
          f"{f4(macro[arm]):>9}"
          f"{f4(cell(1, 'precision', arm)):>8}{f4(cell(1, 'recall', arm)):>8}"
          f"{f4(cell(2, 'precision', arm)):>8}{f4(cell(2, 'recall', arm)):>8}"
          f"{f3(cell(1, 'hd95_mm', arm)):>8}{f3(cell(2, 'hd95_mm', arm)):>8}"
          f"{f3(cell(1, 'asd_mm', arm)):>7}{f3(cell(2, 'asd_mm', arm)):>7}")
print('-' * 128)
print('E1 -> E3 differences (the nested comparison; OBSERVED FACT):')
for label, value in (('macro Dice', macro['delta_vs_e1']),
                     ('PZ Dice   ', cell(2, 'dice', 'e3') - cell(2, 'dice', 'e1')
                      if cell(2, 'dice', 'e1') is not None else None),
                     ('PZ recall ', cell(2, 'recall', 'e3') - cell(2, 'recall', 'e1')
                      if cell(2, 'recall', 'e1') is not None else None),
                     ('CG Dice   ', cell(1, 'dice', 'e3') - cell(1, 'dice', 'e1')
                      if cell(1, 'dice', 'e1') is not None else None)):
    print(f'    {label}: {value:+.4f}' if value is not None else f'    {label}: n/a')
print('=' * 128)
print('NOTE: E1 and E2 vary DIFFERENT factors, so any E2-E3 gap is descriptive context,')
print('      not a ranking. E3 is nested in E1, so E1 is the reference arm.')


In [ ]:
# Paired statistics and the pre-committed verdict, kept strictly separated:
#   OBSERVED FACT      -- the measured numbers
#   STATISTICAL RESULT -- the paired test on n = 20
#   INTERPRETATION     -- the pre-committed category
paired = e3_summary['paired_vs_e1']
outcome = e3_summary['outcome']

print('=' * 86)
print('STATISTICAL RESULT -- paired E3 vs E1 on the same validation cases')
print('=' * 86)
if not paired.get('available'):
    print('  Not available:', paired.get('reason'))
else:
    print(f"  Paired cases: {paired['n_paired_cases']}")
    print(f"  {'Quantity':<18}{'E1':>9}{'E3':>9}{'diff':>10}{'95% CI of diff':>24}"
          f"{'won':>5}{'lost':>6}{'p':>10}")
    print('  ' + '-' * 84)
    for key, label in (('macro_dice', 'macro Dice'),
                       ('class2_dice', 'PZ Dice'),
                       ('class2_recall', 'PZ recall'),
                       ('class2_precision', 'PZ precision'),
                       ('class1_dice', 'CG Dice'),
                       ('class1_recall', 'CG recall'),
                       ('class1_precision', 'CG precision')):
        m = paired['metrics'].get(key)
        if not m:
            continue
        ci = m['difference_bootstrap_ci95']
        ci_text = f"[{ci[0]:+.4f}, {ci[1]:+.4f}]" if ci and ci[0] is not None else 'n/a'
        p = m['wilcoxon']['p_value']
        p_text = ('n/a' if p is None else ('<0.0001' if p < 0.0001 else f'{p:.4f}'))
        print(f"  {label:<18}{m['e1_mean']:>9.4f}{m['e3_mean']:>9.4f}"
              f"{m['mean_difference']:>+10.4f}{ci_text:>24}"
              f"{m['cases_won']:>5}{m['cases_lost']:>6}{p_text:>10}")
    print('  ' + '-' * 84)
    mm = paired['metrics'].get('macro_dice')
    if mm:
        print(f"  Cases won/lost vs E1 on macro Dice: {mm['cases_won']} won / "
              f"{mm['cases_lost']} lost / {mm['cases_tied']} tied")
        print(f"  Wilcoxon signed-rank (two-sided): W = {mm['wilcoxon']['statistic']}, "
              f"p = {p_text}")
    print(f"  n = {paired['n_paired_cases']}; p-values and CIs are DESCRIPTIVE and do not")
    print( '  gate the pre-committed criteria below.')
print('=' * 86)
print()
print('=' * 86)
print('INTERPRETATION -- PRE-COMMITTED success criteria (fixed before this run)')
print('=' * 86)
for label, satisfied in sorted(outcome['checks'].items()):
    if isinstance(satisfied, bool):
        print(f"  [{'YES' if satisfied else ' NO'}] {label}")
print('-' * 86)
for label in ('macro_dice', 'cg_dice', 'pz_recall', 'macro_delta_vs_e1',
              'cg_delta_vs_e1', 'pz_recall_delta_vs_e1'):
    value = outcome['checks'].get(label)
    if isinstance(value, float):
        print(f'  {label:<26}: {value:+.4f}' if 'delta' in label else f'  {label:<26}: {value:.4f}')
print('-' * 86)
print(f"  E3 OUTCOME CATEGORY: {outcome['label']}")
print(f"  Reason             : {outcome['reason']}")
print('=' * 86)
print()
print('E3 is NOT called successful because one metric increased. The category above')
print('is the only verdict asserted, and its thresholds were fixed in advance.')


In [ ]:
from IPython.display import Image, Markdown, display

for name in ('training_loss.png', 'validation_dice.png', 'validation_metrics.png',
             'e3_vs_e1_paired.png'):
    p = os.path.join(E3_DRIVE_DIR, 'plots', name)
    if os.path.isfile(p):
        display(Image(filename=p))

report_path = os.path.join(E3_DRIVE_DIR, 'training_report.md')
if os.path.isfile(report_path):
    display(Markdown(open(report_path, encoding='utf-8').read()))


## 23. Save results / artifact verification

Confirms every required E3 deliverable exists on Drive and is non-empty, so a later
session (or your local VS Code, after syncing Drive) finds a complete run -- and
that no test-split or external-validation artifact was produced.


In [ ]:
REQUIRED = [
    ('best checkpoint',        BEST_CKPT),
    ('latest checkpoint',      LATEST_CKPT),
    ('per-epoch metrics',      METRICS_CSV),
    ('training history',       HISTORY_JSON),
    ('training state',         STATE_JSON),
    ('experiment config',      os.path.join(E3_DRIVE_DIR, 'config.yaml')),
    ('config fingerprint',     os.path.join(E3_DRIVE_DIR, 'config_fingerprint.json')),
    ('per-case metrics',       os.path.join(E3_DRIVE_DIR, 'eval_val_per_case_metrics.csv')),
    ('aggregate summary CSV',  os.path.join(E3_DRIVE_DIR, 'eval_val_summary.csv')),
    ('aggregate summary JSON', os.path.join(E3_DRIVE_DIR, 'eval_val_summary.json')),
    ('experiment summary',     os.path.join(E3_DRIVE_DIR, 'summary.json')),
    ('training report',        os.path.join(E3_DRIVE_DIR, 'training_report.md')),
    ('paired E3-vs-E1 cases',  os.path.join(E3_DRIVE_DIR, 'e3_vs_e1_paired_cases.csv')),
    ('plot: training loss',    os.path.join(E3_DRIVE_DIR, 'plots', 'training_loss.png')),
    ('plot: validation dice',  os.path.join(E3_DRIVE_DIR, 'plots', 'validation_dice.png')),
    ('plot: validation metrics', os.path.join(E3_DRIVE_DIR, 'plots', 'validation_metrics.png')),
    ('plot: paired E3 vs E1',  os.path.join(E3_DRIVE_DIR, 'plots', 'e3_vs_e1_paired.png')),
]

missing = []
print(f"{'Artifact':<26}{'Size':>12}  Path")
print('-' * 110)
for label, path in REQUIRED:
    exists = os.path.isfile(path)
    size = os.path.getsize(path) if exists else 0
    if not exists or size == 0:
        missing.append(label)
    print(f"{label:<26}{(f'{size:,}' if exists else 'MISSING'):>12}  {path}")
print('-' * 110)

# Forbidden artifacts: nothing produced from the test split or external
# validation. Matched on the actual artifact naming (`eval_test_*` /
# `*_test.*`), NOT a bare 'test' substring -- that would flag
# `laTESTt_checkpoint.pt`.
import re as _re
_forbidden_pattern = _re.compile(r'(^|[^a-z])(eval_test|test_split|prostatex|msd)',
                                 _re.IGNORECASE)
forbidden = [f for f in os.listdir(E3_DRIVE_DIR) if _forbidden_pattern.search(f)]
print('Test / external-validation artifacts in the E3 output dir:',
      forbidden or 'NONE (correct)')
assert not forbidden, f'E3 output directory contains forbidden artifacts: {forbidden}'

# The other arms' directories must be untouched by this notebook.
print()
for label, other_dir in (('Baseline', BASELINE_DRIVE_DIR), ('E1', E1_DRIVE_DIR),
                         ('E2', E2_DRIVE_DIR)):
    if os.path.isdir(other_dir):
        print(f'{label} dir present and READ-ONLY to this notebook: {other_dir}')

assert not missing, f'Missing or empty E3 artifacts: {missing}'
print('\nAll required E3 artifacts are present on Drive.')


## 24. Reproducibility metadata

Written next to the results, so the exact commit, environment, seed, loss spec and
resume history behind these numbers stay recoverable from the artifacts alone.


In [ ]:
import json, platform, sys, time

repro = {
    'experiment': EXPERIMENT_NAME,
    'generated_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'git_commit': GIT_COMMIT,
    'git_branch': REPO_BRANCH,
    'repo_url': REPO_URL,
    'notebook': 'notebooks/prostate158_exp_e3_dice_ce_pzweight_colab.ipynb',
    'config_committed': E3_CONFIG_RELPATH,
    'reference_arm': 'exp_e1_dice_ce',
    'independent_variable': {
        'field': 'loss.ce_class_weights',
        'value': list(loss_config.ce_class_weights),
        'class_order': ['background', 'CG', 'PZ'],
        'normalization': 'PyTorch weighted mean (sum w_y l / sum w_y); no manual 1/N',
        'single_factor_vs_e1': True,
    },
    'resume_mode': RESUME,
    'resumed_this_session': bool(RESUME_FROM),
    'training_skipped_already_complete': bool(SKIP_TRAINING),
    'python': sys.version,
    'platform': platform.platform(),
    'torch': torch.__version__,
    'cuda': torch.version.cuda,
    'gpu': GPU_NAME,
    'seed': EXPECTED['seed'],
    'loss': fingerprint['loss_spec'],
    'ce_class_weights': fingerprint['ce_class_weights'],
    'augmentation': fingerprint['augmentation_spec'],
    'epochs_recorded': len(final_history),
    'best_checkpoint_epoch': best_info['epoch'],
    'validation_cases': list(split.val_ids),
    'pre_committed_criteria': E3_CRITERIA,
    'outcome_category': outcome['category'],
    'label_mapping_status': (
        'UNVERIFIED -- class 1 assumed CG, class 2 assumed PZ, identical to '
        'Baseline / E1 / E2. See results/label_mapping/label_mapping_report.md.'
    ),
    'test_split_accessed': False,
    'external_validation_accessed': False,
}
repro_path = os.path.join(E3_DRIVE_DIR, 'reproducibility.json')
with open(repro_path, 'w', encoding='utf-8') as f:
    json.dump(repro, f, indent=2, default=str)
print('Reproducibility record:', repro_path)
print(json.dumps(repro, indent=2, default=str)[:2000])


## 25. Final experiment summary


In [ ]:
macro = e3_summary['macro_dice']
weights = (e3_summary.get('independent_variable') or {}).get('value')

print('=' * 82)
print('EXPERIMENT E3 -- FINAL SUMMARY')
print('=' * 82)
print(f"  Experiment ID           : {e3_summary['experiment']}")
print( '  Design                  : E1 + median-frequency class-weighted CE')
print(f"  Independent variable    : loss.ce_class_weights = {weights}")
print( '                            (index 0 = background, 1 = CG, 2 = PZ)')
print( '  CE normalization        : PyTorch weighted mean; NO manual 1/N rescaling')
print(f"  Git commit              : {e3_summary.get('git_commit')}")
print( '  Dataset                 : Prostate158 (official train archive)')
print( '  Target                  : t2_anatomy_reader1 (0=bg, 1=CG, 2=PZ, UNVERIFIED)')
print( '  Split                   : 119 train / 20 validation (official)')
print( '  Architecture            : 2D U-Net (unchanged from Baseline/E1/E2)')
print( '  Dice term               : 1.0 x SoftDice(CG, PZ), eps 1e-6 (identical to E1)')
print( '  Augmentation            : OFF (identical to E1)')
print( '  Optimizer / LR / WD     : AdamW / 1e-3 / 1e-4')
print( '  Scheduler               : NONE')
print( '  Batch size / Seed       : 8 / 42')
print( '  Precision / AMP         : FP32 / OFF')
print('-' * 82)
print( '  RUN STATUS')
print( '  Maximum epochs          : 100')
print(f"  Epochs completed        : {e3_summary.get('epochs_completed')}")
print(f"  Epoch records on disk   : {e3_summary.get('epoch_records_on_disk')}")
print(f"  Best checkpoint epoch   : {e3_summary.get('best_checkpoint_epoch')}")
print(f"  Best val Dice proxy     : {e3_summary.get('best_val_dice_proxy')}")
print(f'  Resumed this session    : {bool(RESUME_FROM)}')
print(f'  Training skipped (done) : {bool(SKIP_TRAINING)}')
print('-' * 82)
print( '  VALIDATION RESULTS (OBSERVED FACT, 20 cases)')
for cid, label in (('1', 'CG'), ('2', 'PZ')):
    for metric in ('dice', 'recall', 'precision'):
        c = e3_summary['per_class'][cid][metric]
        e1v = '   n/a' if c['e1'] is None else f"{c['e1']:.4f}"
        print(f"  {label} {metric:<10}: baseline {c['baseline']:.4f} | E1 {e1v} | "
              f"E2 {c['e2']:.4f} | E3 {c['e3']:.4f}  ({c['delta_vs_e1']:+.4f} vs E1)")
print(f"  MACRO Dice    : baseline {macro['baseline']:.4f} | E1 {macro['e1']:.4f} | "
      f"E2 {macro['e2']:.4f} | E3 {macro['e3']:.4f}  ({macro['delta_vs_e1']:+.4f} vs E1)")
print('-' * 82)
print( '  STATISTICAL RESULT')
mm = (e3_summary['paired_vs_e1'].get('metrics') or {}).get('macro_dice')
if mm:
    p = mm['wilcoxon']['p_value']
    print(f"  Paired macro Dice vs E1 : {mm['cases_won']} won / {mm['cases_lost']} lost / "
          f"{mm['cases_tied']} tied (n={mm['n']})")
    print(f"  Wilcoxon signed-rank    : W={mm['wilcoxon']['statistic']}, "
          f"p={'n/a' if p is None else format(p, '.4f')} (descriptive, n={mm['n']})")
else:
    print('  Paired comparison       : not available (see section 22)')
print('-' * 82)
print( '  INTERPRETATION (pre-committed)')
print(f"  E3 OUTCOME              : {outcome['label']}")
print(f"  Reason                  : {outcome['reason']}")
print('-' * 82)
print( '  Validation cases        : 20')
print( '  Official test cases used: 0')
print( '  External validation     : none')
print( '  Label mapping           : UNVERIFIED (recorded as a limitation in the report)')
print('=' * 82)
print()
print('Artifacts on Drive:', E3_DRIVE_DIR)
for entry in sorted(os.listdir(E3_DRIVE_DIR)):
    full = os.path.join(E3_DRIVE_DIR, entry)
    marker = '/' if os.path.isdir(full) else ''
    print(f'    {entry}{marker}')
print()
print('The official 19-case test set was NOT used during E3 training, model')
print('selection, or experiment comparison. No external validation was accessed.')
print('No final model was selected. No post-processing or TTA was applied.')
print()
print('NEXT STEPS (not done here):')
print('  1. Report E3 alongside the Baseline, E1 and E2 to the supervisor.')
print('  2. Resolve the CG/PZ label-mapping question before the thesis writes')
print('     anatomical conclusions about either zone.')
print('  3. Complete RQ-1 selection across all arms.')
print('  4. Only after that: a SINGLE official 19-case test evaluation of the')
print('     selected configuration.')
